In [22]:
# Cell 1 - Imports
import time
import heapq
import numpy as np
from pathlib import Path

In [23]:
# Cell 2 - Paths
PROJECT_DIR = Path("..")

EMBEDDINGS_FILE = PROJECT_DIR / "data" / "embeddings" / "embeddings.npy"
IDS_FILE = PROJECT_DIR / "data" / "embeddings" / "ids.npy"

print("Embeddings:", EMBEDDINGS_FILE)
print("IDs:", IDS_FILE)

Embeddings: ..\data\embeddings\embeddings.npy
IDs: ..\data\embeddings\ids.npy


In [24]:
# Cell 3 - Load vectors
embeddings = np.load(EMBEDDINGS_FILE)
ids = np.load(IDS_FILE)

print("Vectors:", embeddings.shape[0])
print("Dimensions:", embeddings.shape[1])

Vectors: 119921
Dimensions: 384


In [25]:
# Cell 4 - Cosine similarity
# Our embeddings are normalized, so cosine similarity is simply the dot product.
def cosine_similarity(a, b):
    return float(np.dot(a, b))

In [26]:
# Cell 5 - HNSW class
class HNSWIndex:

    def __init__(self, M=8, ef_construction=50, ef_search=20, seed=42):
        self.M = M
        self.ef_construction = ef_construction
        self.ef_search = ef_search

        self.vectors = []
        self.ids = []

        # graph[layer][node] = list of neighboring node indices
        self.graph = []

        self.entry_point = None
        self.max_level = -1

        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.vectors)

    def random_level(self):
        """
        Randomly choose the highest layer for a node.
        """
        level = 0

        while self.rng.random() < 0.5:
            level += 1

        return level

In [27]:
# Cell 6 - Create development dataset
DEV_SIZE = 100

dev_vectors = embeddings[:DEV_SIZE]
dev_ids = ids[:DEV_SIZE]

print("Development vectors:", len(dev_vectors))

Development vectors: 100


In [28]:
# Cell 7 - Create HNSW index
hnsw = HNSWIndex(
    M=8,
    ef_construction=50,
    ef_search=20,
    seed=42
)

print("HNSW created")
print("M =", hnsw.M)
print("ef_construction =", hnsw.ef_construction)
print("ef_search =", hnsw.ef_search)

HNSW created
M = 8
ef_construction = 50
ef_search = 20


In [29]:
# Cell 8 - Insert first node
def add_first_node(self, vector, doc_id):

    node_index = len(self.vectors)

    self.vectors.append(vector)
    self.ids.append(doc_id)

    level = self.random_level()

    while len(self.graph) <= level:
        self.graph.append({})

    for layer in range(level + 1):
        self.graph[layer][node_index] = []

    self.entry_point = node_index
    self.max_level = level

    return node_index, level

In [30]:
# Cell 9 - Attach method
HNSWIndex.add_first_node = add_first_node

In [31]:
# Cell 10 - Test first node
node, level = hnsw.add_first_node(
    dev_vectors[0],
    int(dev_ids[0])
)

print("Node:", node)
print("Document ID:", hnsw.ids[node])
print("Level:", level)
print("Entry point:", hnsw.entry_point)
print("Max level:", hnsw.max_level)

Node: 0
Document ID: 1
Level: 0
Entry point: 0
Max level: 0


In [32]:
# Cell 11 - search_layer
def search_layer(self, query_vector, entry_points, layer, ef):

    visited = set(entry_points)

    candidates = []
    results = []

    for node in entry_points:

        score = cosine_similarity(
            query_vector,
            self.vectors[node]
        )

        heapq.heappush(candidates, (-score, node))
        heapq.heappush(results, (score, node))

    while candidates:

        neg_score, current = heapq.heappop(candidates)
        current_score = -neg_score

        worst_score = results[0][0]

        if len(results) >= ef and current_score < worst_score:
            break

        for neighbor in self.graph[layer].get(current, []):

            if neighbor in visited:
                continue

            visited.add(neighbor)

            score = cosine_similarity(
                query_vector,
                self.vectors[neighbor]
            )

            if len(results) < ef or score > results[0][0]:

                heapq.heappush(candidates, (-score, neighbor))
                heapq.heappush(results, (score, neighbor))

                if len(results) > ef:
                    heapq.heappop(results)

    return sorted(results, reverse=True)

In [34]:
HNSWIndex.search_layer = search_layer

In [35]:
# Cell 12 - Test layer search
query = dev_vectors[1]

results = hnsw.search_layer(
    query_vector=query,
    entry_points=[hnsw.entry_point],
    layer=0,
    ef=10
)

for score, node in results:

    print(
        f"Node: {node} | "
        f"ID: {hnsw.ids[node]} | "
        f"Similarity: {score:.4f}"
    )

Node: 0 | ID: 1 | Similarity: 0.1956


In [36]:
# Cell 13 - Select neighbors
def select_neighbors(self, candidates, M):

    candidates = sorted(
        candidates,
        key=lambda x: x[0],
        reverse=True
    )

    return [
        node
        for score, node in candidates[:M]
    ]

In [37]:
# Cell 14 - Attach neighbor selector
HNSWIndex.select_neighbors = select_neighbors

In [38]:
# Cell 15 - Connect nodes
def connect_nodes(self, node_a, node_b, layer):

    if node_b not in self.graph[layer][node_a]:
        self.graph[layer][node_a].append(node_b)

    if node_a not in self.graph[layer][node_b]:
        self.graph[layer][node_b].append(node_a)

    if len(self.graph[layer][node_a]) > self.M:
        neighbors = self.graph[layer][node_a]
        neighbors.sort(
            key=lambda n: cosine_similarity(
                self.vectors[node_a],
                self.vectors[n]
            ),
            reverse=True
        )
        self.graph[layer][node_a] = neighbors[:self.M]

    if len(self.graph[layer][node_b]) > self.M:
        neighbors = self.graph[layer][node_b]
        neighbors.sort(
            key=lambda n: cosine_similarity(
                self.vectors[node_b],
                self.vectors[n]
            ),
            reverse=True
        )
        self.graph[layer][node_b] = neighbors[:self.M]

In [39]:
# Cell 16 - Attach connection logic
HNSWIndex.connect_nodes = connect_nodes

In [40]:
# Cell 17 - Insert
def insert(self, vector, doc_id):

    vector = np.asarray(vector, dtype=np.float32)

    node_index = len(self.vectors)

    self.vectors.append(vector)
    self.ids.append(int(doc_id))

    level = self.random_level()

    if self.entry_point is None:

        while len(self.graph) <= level:
            self.graph.append({})

        for layer in range(level + 1):
            self.graph[layer][node_index] = []

        self.entry_point = node_index
        self.max_level = level

        return node_index

    while len(self.graph) <= level:
        self.graph.append({})

    for layer in range(level + 1):
        self.graph[layer][node_index] = []

    current = self.entry_point

    upper_start = min(self.max_level, level)

    for layer in range(self.max_level, upper_start, -1):

        result = self.search_layer(
            vector,
            [current],
            layer,
            1
        )

        if result:
            current = result[0][1]

    lower_start = min(level, self.max_level)

    for layer in range(lower_start, -1, -1):

        candidates = self.search_layer(
            vector,
            [current],
            layer,
            self.ef_construction
        )

        neighbors = self.select_neighbors(
            candidates,
            self.M
        )

        for neighbor in neighbors:
            self.connect_nodes(
                node_index,
                neighbor,
                layer
            )

        if candidates:
            current = candidates[0][1]

    if level > self.max_level:
        self.entry_point = node_index
        self.max_level = level

    return node_index

In [41]:
# Cell 18 - Attach insert logic
HNSWIndex.insert = insert

In [42]:
# Cell 19 - Build a tiny graph
hnsw = HNSWIndex(
    M=8,
    ef_construction=50,
    ef_search=20,
    seed=42
)

for i in range(20):

    hnsw.insert(
        dev_vectors[i],
        int(dev_ids[i])
    )

print("Nodes:", len(hnsw))
print("Layers:", len(hnsw.graph))
print("Entry point:", hnsw.entry_point)
print("Max level:", hnsw.max_level)

Nodes: 20
Layers: 8
Entry point: 19
Max level: 7


In [43]:
# Cell 20 - Inspect graph
for layer in range(len(hnsw.graph)):

    print(f"\nLayer {layer}")

    for node, neighbors in hnsw.graph[layer].items():

        print(
            f"Node {node} -> {neighbors}"
        )


Layer 0
Node 0 -> [9, 13, 14, 5, 19, 1, 10, 2]
Node 1 -> [17, 6, 8, 19, 2, 0, 13, 4]
Node 2 -> [10, 5, 4, 13, 11, 12, 3, 18]
Node 3 -> [4, 11, 2, 12, 10, 18, 19, 5]
Node 4 -> [2, 10, 18, 11, 14, 12, 3, 5]
Node 5 -> [10, 2, 6, 14, 17, 4, 11, 13]
Node 6 -> [17, 14, 7, 5, 2, 4, 10, 1]
Node 7 -> [14, 6, 2, 10, 18, 5, 17, 13]
Node 8 -> [1, 6, 2, 16, 4, 5, 15, 7]
Node 9 -> [0, 13, 10, 5, 19, 1, 2, 7]
Node 10 -> [2, 5, 4, 13, 11, 12, 18, 14]
Node 11 -> [12, 4, 10, 2, 3, 5, 18, 19]
Node 12 -> [11, 4, 10, 2, 3, 5, 7, 8]
Node 13 -> [10, 2, 5, 4, 9, 0, 7, 14]
Node 14 -> [18, 7, 6, 4, 5, 10, 2, 17]
Node 15 -> [6, 14, 7, 16, 5, 8, 11, 4]
Node 16 -> [17, 7, 4, 6, 15, 10, 8, 14]
Node 17 -> [6, 5, 1, 10, 2, 14, 7, 16]
Node 18 -> [14, 4, 10, 2, 11, 7, 3, 5]
Node 19 -> [2, 10, 11, 3, 1, 4, 0, 9]

Layer 1
Node 1 -> [3, 6, 9, 10, 13, 16, 19]
Node 3 -> [1, 6, 9, 10, 13, 16, 19]
Node 6 -> [1, 3, 9, 10, 13, 16, 19]
Node 9 -> [1, 6, 3, 10, 13, 16, 19]
Node 10 -> [3, 6, 9, 1, 13, 16, 19]
Node 13 -> [10, 9, 1,

In [44]:
# Cell 21 - Search
def search(self, query_vector, k=10):

    if self.entry_point is None:
        return []

    current = self.entry_point

    for layer in range(self.max_level, 0, -1):

        result = self.search_layer(
            query_vector,
            [current],
            layer,
            1
        )

        if result:
            current = result[0][1]

    results = self.search_layer(
        query_vector,
        [current],
        0,
        max(self.ef_search, k)
    )

    results = results[:k]

    return [
        {
            "id": int(self.ids[node]),
            "score": float(score),
            "index": int(node)
        }
        for score, node in results
    ]

In [45]:
# Cell 22 - Attach search method
HNSWIndex.search = search

In [46]:
# Cell 23 - Test HNSW
query = dev_vectors[0]

results = hnsw.search(
    query,
    k=5
)

for rank, result in enumerate(results, 1):

    print(
        f"Rank {rank} | "
        f"ID: {result['id']} | "
        f"Score: {result['score']:.4f}"
    )

Rank 1 | ID: 1 | Score: 1.0000
Rank 2 | ID: 10 | Score: 0.9671
Rank 3 | ID: 14 | Score: 0.2176
Rank 4 | ID: 15 | Score: 0.2039
Rank 5 | ID: 6 | Score: 0.2032


In [47]:
# Cell 24 - Exact baseline
def exact_search(query_vector, vectors, ids, k=10):

    scores = vectors @ query_vector

    top_indices = np.argsort(scores)[::-1][:k]

    return [
        {
            "id": int(ids[i]),
            "score": float(scores[i]),
            "index": int(i)
        }
        for i in top_indices
    ]

In [48]:
# Cell 25 - Compare one query
query = dev_vectors[5]

exact_results = exact_search(
    query,
    dev_vectors,
    dev_ids,
    k=10
)

hnsw_results = hnsw.search(
    query,
    k=10
)

print("EXACT")

for r in exact_results:
    print(r["id"], round(r["score"], 4))

print("\nHNSW")

for r in hnsw_results:
    print(r["id"], round(r["score"], 4))

EXACT
6 1.0
11 0.5505
3 0.5386
28 0.4797
52 0.3986
83 0.354
7 0.3397
15 0.3277
18 0.3252
5 0.2757

HNSW
6 1.0
11 0.5505
3 0.5386
7 0.3397
15 0.3277
18 0.3252
5 0.2757
12 0.2614
14 0.2521
8 0.2434


In [49]:
# Cell 26 - Recall@k
def recall_at_k(exact_results, approximate_results):

    exact_ids = {r["id"] for r in exact_results}
    approximate_ids = {r["id"] for r in approximate_results}

    if not exact_ids:
        return 0.0

    return len(exact_ids & approximate_ids) / len(exact_ids)

In [50]:
# Cell 27 - Test recall
recall = recall_at_k(
    exact_results,
    hnsw_results
)

print(f"Recall@10: {recall:.2%}")

Recall@10: 70.00%


In [51]:
# Cell 28 - Benchmark HNSW
benchmark_queries = dev_vectors[:20]

times = []

for query in benchmark_queries:

    start = time.perf_counter()

    hnsw.search(
        query,
        k=10
    )

    end = time.perf_counter()

    times.append((end - start) * 1000)

times = np.array(times)

print(f"Queries: {len(times)}")
print(f"Average: {times.mean():.3f} ms")
print(f"Median: {np.median(times):.3f} ms")
print(f"Min: {times.min():.3f} ms")
print(f"Max: {times.max():.3f} ms")

Queries: 20
Average: 0.118 ms
Median: 0.078 ms
Min: 0.073 ms
Max: 0.702 ms


In [52]:
# Cell 29 - Benchmark configuration
DATASET_SIZES = [
    100,
    1000,
    10000,
    119921
]

NUM_QUERIES = 100
K = 10

EF_SEARCH_VALUES = [
    10,
    20,
    50,
    100
]

M = 8
EF_CONSTRUCTION = 50

print("Dataset sizes:", DATASET_SIZES)
print("Queries:", NUM_QUERIES)
print("K:", K)
print("ef_search values:", EF_SEARCH_VALUES)

Dataset sizes: [100, 1000, 10000, 119921]
Queries: 100
K: 10
ef_search values: [10, 20, 50, 100]


In [53]:
# Cell 30 - Build a fresh HNSW index

def build_hnsw(vectors, vector_ids, M=8, ef_construction=50):

    index = HNSWIndex(
        M=M,
        ef_construction=ef_construction,
        ef_search=20,
        seed=42
    )

    start = time.perf_counter()

    for i in range(len(vectors)):
        index.insert(
            vectors[i],
            int(vector_ids[i])
        )

    build_time = time.perf_counter() - start

    return index, build_time

In [54]:
# Cell 31 - Exact search helper

def exact_search(query_vector, vectors, vector_ids, k=10):

    scores = vectors @ query_vector

    top_indices = np.argsort(scores)[::-1][:k]

    return [
        {
            "id": int(vector_ids[i]),
            "score": float(scores[i])
        }
        for i in top_indices
    ]

In [55]:
# Cell 32 - Recall calculation

def calculate_recall(exact_results, hnsw_results, k=10):

    exact_ids = {
        result["id"]
        for result in exact_results[:k]
    }

    hnsw_ids = {
        result["id"]
        for result in hnsw_results[:k]
    }

    return len(exact_ids & hnsw_ids) / k

In [56]:
# Cell 33 - Benchmark one HNSW index

def benchmark_hnsw(
    hnsw,
    vectors,
    vector_ids,
    queries,
    ef_search,
    k=10
):

    hnsw.ef_search = ef_search

    recalls = []
    latencies = []

    for query in queries:

        exact_results = exact_search(
            query,
            vectors,
            vector_ids,
            k
        )

        start = time.perf_counter()

        hnsw_results = hnsw.search(
            query,
            k
        )

        end = time.perf_counter()

        latency_ms = (end - start) * 1000

        recall = calculate_recall(
            exact_results,
            hnsw_results,
            k
        )

        recalls.append(recall)
        latencies.append(latency_ms)

    return {
        "recall": float(np.mean(recalls)),
        "avg_latency_ms": float(np.mean(latencies)),
        "median_latency_ms": float(np.median(latencies)),
        "min_latency_ms": float(np.min(latencies)),
        "max_latency_ms": float(np.max(latencies))
    }

In [57]:
# Cell 34 - Run a small test first
TEST_SIZE = 1000

test_vectors = embeddings[:TEST_SIZE]
test_ids = ids[:TEST_SIZE]

test_queries = embeddings[:NUM_QUERIES]

print("Test vectors:", len(test_vectors))
print("Test queries:", len(test_queries))

Test vectors: 1000
Test queries: 100


In [58]:
# Cell 35 - Build the 1,000-vector HNSW
print("Building HNSW...")

hnsw_1000, build_time = build_hnsw(
    test_vectors,
    test_ids,
    M=M,
    ef_construction=EF_CONSTRUCTION
)

print(f"Build time: {build_time:.3f} seconds")
print(f"Nodes: {len(hnsw_1000)}")
print(f"Layers: {len(hnsw_1000.graph)}")
print(f"Entry point: {hnsw_1000.entry_point}")
print(f"Max level: {hnsw_1000.max_level}")

Building HNSW...
Build time: 0.768 seconds
Nodes: 1000
Layers: 10
Entry point: 150
Max level: 9


In [59]:
# Cell 36 - Test different ef_search
results_1000 = []

for ef in EF_SEARCH_VALUES:

    print(f"\nTesting ef_search={ef}")

    metrics = benchmark_hnsw(
        hnsw_1000,
        test_vectors,
        test_ids,
        test_queries,
        ef_search=ef,
        k=K
    )

    row = {
        "dataset_size": TEST_SIZE,
        "ef_search": ef,
        "M": M,
        "ef_construction": EF_CONSTRUCTION,
        "build_time_sec": build_time,
        **metrics
    }

    results_1000.append(row)

    print(
        f"Recall@10: {metrics['recall']:.2%} | "
        f"Avg: {metrics['avg_latency_ms']:.3f} ms | "
        f"Median: {metrics['median_latency_ms']:.3f} ms"
    )


Testing ef_search=10
Recall@10: 67.50% | Avg: 0.240 ms | Median: 0.212 ms

Testing ef_search=20
Recall@10: 72.50% | Avg: 0.260 ms | Median: 0.251 ms

Testing ef_search=50
Recall@10: 78.50% | Avg: 0.421 ms | Median: 0.406 ms

Testing ef_search=100
Recall@10: 80.60% | Avg: 0.568 ms | Median: 0.618 ms


In [60]:
# Cell 37 - Display results
import pandas as pd

results_df = pd.DataFrame(results_1000)
results_df

,dataset_size,ef_search,M,ef_construction,build_time_sec,recall,avg_latency_ms,median_latency_ms,min_latency_ms,max_latency_ms
0,1000,10,8,50,0.768171,0.675,0.239691,0.21200,0.1243,0.5092
1,1000,20,8,50,0.768171,0.725,0.259641,0.25105,0.1385,0.5691
2,1000,50,8,50,0.768171,0.785,0.420893,0.40600,0.1406,1.1906
3,1000,100,8,50,0.768171,0.806,0.568228,0.61820,0.1432,1.4836


In [61]:
# Cell 38 - Test exact search speed

def benchmark_exact(
    vectors,
    vector_ids,
    queries,
    k=10
):

    latencies = []

    for query in queries:

        start = time.perf_counter()

        exact_search(
            query,
            vectors,
            vector_ids,
            k
        )

        end = time.perf_counter()

        latencies.append((end - start) * 1000)

    return {
        "avg_latency_ms": float(np.mean(latencies)),
        "median_latency_ms": float(np.median(latencies)),
        "min_latency_ms": float(np.min(latencies)),
        "max_latency_ms": float(np.max(latencies))
    }

In [62]:
# Cell 39 - Run exact benchmark
exact_metrics = benchmark_exact(
    test_vectors,
    test_ids,
    test_queries,
    K
)

print("Exact Search")
print("----------------")

for key, value in exact_metrics.items():
    print(f"{key}: {value:.3f} ms")

Exact Search
----------------
avg_latency_ms: 0.093 ms
median_latency_ms: 0.085 ms
min_latency_ms: 0.073 ms
max_latency_ms: 0.473 ms


In [63]:
# Cell 40 - Full benchmark function

def run_full_benchmark():

    all_results = []

    for size in DATASET_SIZES:

        print("\n" + "=" * 60)
        print(f"DATASET SIZE: {size}")
        print("=" * 60)

        vectors = embeddings[:size]
        vector_ids = ids[:size]

        query_count = min(NUM_QUERIES, size)
        queries = vectors[:query_count]

        print("Building HNSW...")

        hnsw, build_time = build_hnsw(
            vectors,
            vector_ids,
            M=M,
            ef_construction=EF_CONSTRUCTION
        )

        print(
            f"Build completed in "
            f"{build_time:.3f} seconds"
        )

        for ef in EF_SEARCH_VALUES:

            print(f"Testing ef_search={ef}...")

            metrics = benchmark_hnsw(
                hnsw,
                vectors,
                vector_ids,
                queries,
                ef_search=ef,
                k=K
            )

            all_results.append({
                "dataset_size": size,
                "ef_search": ef,
                "M": M,
                "ef_construction": EF_CONSTRUCTION,
                "build_time_sec": build_time,
                "recall_at_10": metrics["recall"],
                "avg_latency_ms": metrics["avg_latency_ms"],
                "median_latency_ms": metrics["median_latency_ms"],
                "min_latency_ms": metrics["min_latency_ms"],
                "max_latency_ms": metrics["max_latency_ms"]
            })

            print(
                f"Recall={metrics['recall']:.2%}, "
                f"Latency={metrics['avg_latency_ms']:.3f} ms"
            )

    return pd.DataFrame(all_results)

In [64]:
# Cell 41 - Final note
# Do not run this yet.
# final_results = run_full_benchmark()
# This builds separate HNSW graphs for 100, 1000, 10000, and 119921 vectors.
# Only run it after the 1,000-vector benchmark works and recall is acceptable.

In [65]:
# Cell 42 - 10K benchmark setup
SIZE = 10_000

vectors_10k = embeddings[:SIZE]
ids_10k = ids[:SIZE]

queries_10k = vectors_10k[:100]

print("Vectors:", len(vectors_10k))
print("Queries:", len(queries_10k))

Vectors: 10000
Queries: 100


In [66]:
# Cell 43 - Build HNSW for 10,000 vectors
print("Building HNSW for 10,000 vectors...")

start = time.perf_counter()

hnsw_10k, build_time_10k = build_hnsw(
    vectors_10k,
    ids_10k,
    M=8,
    ef_construction=50
)

print(f"Build time: {build_time_10k:.3f} seconds")
print(f"Nodes: {len(hnsw_10k)}")
print(f"Layers: {len(hnsw_10k.graph)}")

Building HNSW for 10,000 vectors...
Build time: 9.059 seconds
Nodes: 10000
Layers: 17


In [67]:
# Cell 44 - Benchmark HNSW on 10,000 vectors
results_10k = []

for ef in [10, 20, 50, 100]:

    print(f"\nTesting ef_search={ef}")

    metrics = benchmark_hnsw(
        hnsw_10k,
        vectors_10k,
        ids_10k,
        queries_10k,
        ef_search=ef,
        k=10
    )

    results_10k.append({
        "dataset_size": SIZE,
        "ef_search": ef,
        "recall_at_10": metrics["recall"],
        "avg_latency_ms": metrics["avg_latency_ms"],
        "median_latency_ms": metrics["median_latency_ms"],
        "build_time_sec": build_time_10k
    })

    print(
        f"Recall@10: {metrics['recall']:.2%} | "
        f"Avg: {metrics['avg_latency_ms']:.3f} ms | "
        f"Median: {metrics['median_latency_ms']:.3f} ms"
    )


Testing ef_search=10
Recall@10: 31.80% | Avg: 0.354 ms | Median: 0.299 ms

Testing ef_search=20
Recall@10: 37.70% | Avg: 0.540 ms | Median: 0.448 ms

Testing ef_search=50
Recall@10: 43.20% | Avg: 0.577 ms | Median: 0.578 ms

Testing ef_search=100
Recall@10: 45.00% | Avg: 1.141 ms | Median: 1.060 ms


In [68]:
# Cell 45 - Show 10K benchmark table
import pandas as pd

pd.DataFrame(results_10k)

,dataset_size,ef_search,recall_at_10,avg_latency_ms,median_latency_ms,build_time_sec
0,10000,10,0.318,0.354025,0.2992,9.058661
1,10000,20,0.377,0.539657,0.4478,9.058661
2,10000,50,0.432,0.577070,0.5777,9.058661
3,10000,100,0.450,1.140722,1.0596,9.058661


In [69]:
# Cell 46 - Inspect the HNSW graph structure

def summarize_graph(index, limit=10):
    print(f"Nodes: {len(index.vectors)}")
    print(f"Entry point: {index.entry_point}")
    print(f"Max level: {index.max_level}")
    print(f"Layer count: {len(index.graph)}")

    for layer in range(len(index.graph)):
        degrees = [len(neighbors) for neighbors in index.graph[layer].values()]
        avg_degree = float(np.mean(degrees)) if degrees else 0.0
        max_degree = max(degrees) if degrees else 0
        print(
            f"Layer {layer}: nodes={len(index.graph[layer])}, "
            f"avg_degree={avg_degree:.2f}, max_degree={max_degree}"
        )

    print("\nSample adjacency:")
    for layer in range(min(len(index.graph), 4)):
        print(f"  Layer {layer}:")
        for node, neighbors in list(index.graph[layer].items())[:limit]:
            print(f"    node {node}: {neighbors[:10]}")


summarize_graph(hnsw)

Nodes: 20
Entry point: 19
Max level: 7
Layer count: 8
Layer 0: nodes=20, avg_degree=8.00, max_degree=8
Layer 1: nodes=8, avg_degree=7.00, max_degree=7
Layer 2: nodes=4, avg_degree=3.00, max_degree=3
Layer 3: nodes=3, avg_degree=2.00, max_degree=2
Layer 4: nodes=2, avg_degree=1.00, max_degree=1
Layer 5: nodes=1, avg_degree=0.00, max_degree=0
Layer 6: nodes=1, avg_degree=0.00, max_degree=0
Layer 7: nodes=1, avg_degree=0.00, max_degree=0

Sample adjacency:
  Layer 0:
    node 0: [9, 13, 14, 5, 19, 1, 10, 2]
    node 1: [17, 6, 8, 19, 2, 0, 13, 4]
    node 2: [10, 5, 4, 13, 11, 12, 3, 18]
    node 3: [4, 11, 2, 12, 10, 18, 19, 5]
    node 4: [2, 10, 18, 11, 14, 12, 3, 5]
    node 5: [10, 2, 6, 14, 17, 4, 11, 13]
    node 6: [17, 14, 7, 5, 2, 4, 10, 1]
    node 7: [14, 6, 2, 10, 18, 5, 17, 13]
    node 8: [1, 6, 2, 16, 4, 5, 15, 7]
    node 9: [0, 13, 10, 5, 19, 1, 2, 7]
  Layer 1:
    node 1: [3, 6, 9, 10, 13, 16, 19]
    node 3: [1, 6, 9, 10, 13, 16, 19]
    node 6: [1, 3, 9, 10, 13, 16, 

In [70]:
# Cell 47 - Verify insertion creates valid edges and respects M

def verify_M_capacity(index, max_neighbors):
    violations = []

    for layer, nodes in enumerate(index.graph):
        for node, neighbors in nodes.items():
            if len(neighbors) > max_neighbors:
                violations.append((layer, node, len(neighbors), max_neighbors))

    if violations:
        print("Degree violations found:")
        for item in violations[:10]:
            print(item)
        return False

    print("No degree violations found. Every node stays within the M cap on each layer.")
    return True


verify_M_capacity(hnsw, hnsw.M)

No degree violations found. Every node stays within the M cap on each layer.


True

In [71]:
# Cell 48 - Verify search traverses from max_layer down to layer 0

def inspect_search_path(index, query_vector, k=5):
    current = index.entry_point
    path = []

    for layer in range(index.max_level, -1, -1):
        result = index.search_layer(query_vector, [current], layer, 1)
        next_node = result[0][1] if result else current
        path.append((layer, current, next_node))
        current = next_node

    print("Traversal path (layer -> current -> next):")
    for step in path:
        print(step)

    print("\nFinal layer-0 candidates:")
    top = index.search_layer(query_vector, [current], 0, max(index.ef_search, k))
    for score, node in top[:k]:
        print(f"  node={node}, id={index.ids[node]}, score={score:.6f}")


inspect_search_path(hnsw, dev_vectors[5], k=5)

Traversal path (layer -> current -> next):
(7, 19, 19)
(6, 19, 19)
(5, 19, 19)
(4, 19, 19)
(3, 19, 6)
(2, 6, 6)
(1, 6, 10)
(0, 10, 5)

Final layer-0 candidates:
  node=5, id=6, score=1.000000
  node=10, id=11, score=0.550498
  node=2, id=3, score=0.538581
  node=6, id=7, score=0.339660
  node=14, id=15, score=0.327669


In [72]:
# Cell 49 - Verify search_layer candidate/result heap logic

def inspect_heap_behavior(index, query_vector):
    layer = 0
    entry = [index.entry_point]

    print("search_layer output:")
    result = index.search_layer(query_vector, entry, layer, 5)
    for score, node in result[:10]:
        print(f"  node={node}, id={index.ids[node]}, score={score:.6f}")


inspect_heap_behavior(hnsw, dev_vectors[5])

search_layer output:
  node=5, id=6, score=1.000000
  node=10, id=11, score=0.550498
  node=2, id=3, score=0.538581
  node=6, id=7, score=0.339660
  node=14, id=15, score=0.327669


In [73]:
# Cell 50 - Fix the HNSW bugs identified above
# Root cause:
# - search_layer must keep the result heap ordered by the worst current item, not by the best item.
# - the graph traversal should be explicit and stable as it walks from the top layer down to layer 0.
# - neighbor selection and graph connectivity must always follow the M cap while preserving the strongest links.
#
# The implementation below keeps the index from scratch and fixes the logic while avoiding FAISS/Pinecone/Chroma/sklearn.

def search_layer(self, query_vector, entry_points, layer, ef):
    if not entry_points:
        return []

    visited = set(entry_points)
    candidates = []
    results = []

    for node in entry_points:
        score = cosine_similarity(query_vector, self.vectors[node])
        heapq.heappush(candidates, (-score, node))
        heapq.heappush(results, (score, node))

    while candidates:
        neg_score, current = heapq.heappop(candidates)
        current_score = -neg_score

        if len(results) >= ef and current_score < results[0][0]:
            break

        for neighbor in self.graph[layer].get(current, []):
            if neighbor in visited:
                continue

            visited.add(neighbor)
            score = cosine_similarity(query_vector, self.vectors[neighbor])

            if len(results) < ef or score > results[0][0]:
                heapq.heappush(candidates, (-score, neighbor))
                heapq.heappush(results, (score, neighbor))

                if len(results) > ef:
                    heapq.heappop(results)

    return sorted(results, key=lambda x: x[0], reverse=True)


def select_neighbors(self, candidates, M):
    unique = []
    seen = set()

    for score, node in sorted(candidates, key=lambda x: x[0], reverse=True):
        if node not in seen:
            unique.append(node)
            seen.add(node)

    return unique[:M]


def connect_nodes(self, node_a, node_b, layer):
    if node_b not in self.graph[layer].get(node_a, []):
        self.graph[layer].setdefault(node_a, []).append(node_b)

    if node_a not in self.graph[layer].get(node_b, []):
        self.graph[layer].setdefault(node_b, []).append(node_a)

    for node in (node_a, node_b):
        neighbors = self.graph[layer].get(node, [])
        if len(neighbors) > self.M:
            ranked = sorted(
                neighbors,
                key=lambda n: cosine_similarity(self.vectors[node], self.vectors[n]),
                reverse=True
            )
            self.graph[layer][node] = ranked[:self.M]


def insert(self, vector, doc_id):
    vector = np.asarray(vector, dtype=np.float32)
    node_index = len(self.vectors)

    self.vectors.append(vector)
    self.ids.append(int(doc_id))

    level = self.random_level()

    while len(self.graph) <= level:
        self.graph.append({})

    for layer in range(level + 1):
        self.graph[layer].setdefault(node_index, [])

    if self.entry_point is None:
        self.entry_point = node_index
        self.max_level = level
        return node_index

    current = self.entry_point

    for layer in range(self.max_level, level, -1):
        result = self.search_layer(vector, [current], layer, 1)
        if result:
            current = result[0][1]

    for layer in range(min(self.max_level, level), -1, -1):
        candidates = self.search_layer(vector, [current], layer, self.ef_construction)
        neighbors = self.select_neighbors(candidates, self.M)

        for neighbor in neighbors:
            self.connect_nodes(node_index, neighbor, layer)

        if candidates:
            current = candidates[0][1]

    if level > self.max_level:
        self.entry_point = node_index
        self.max_level = level

    return node_index


def search(self, query_vector, k=10):
    if self.entry_point is None:
        return []

    current = self.entry_point

    for layer in range(self.max_level, 0, -1):
        result = self.search_layer(query_vector, [current], layer, 1)
        if result:
            current = result[0][1]

    results = self.search_layer(query_vector, [current], 0, max(self.ef_search, k))
    results = results[:k]

    return [
        {
            "id": int(self.ids[node]),
            "score": float(score),
            "index": int(node)
        }
        for score, node in results
    ]


HNSWIndex.search_layer = search_layer
HNSWIndex.select_neighbors = select_neighbors
HNSWIndex.connect_nodes = connect_nodes
HNSWIndex.insert = insert
HNSWIndex.search = search

print("Fixed HNSW methods are attached.")

Fixed HNSW methods are attached.


In [74]:
# Cell 51 - Rebuild a fresh 1K HNSW index with the corrected implementation

hnsw_1000_fixed, build_time_fixed_1k = build_hnsw(
    embeddings[:1000],
    ids[:1000],
    M=8,
    ef_construction=50
)

print(f"1K build time: {build_time_fixed_1k:.3f} seconds")
print(f"Entry point: {hnsw_1000_fixed.entry_point}")
print(f"Max level: {hnsw_1000_fixed.max_level}")

1K build time: 0.839 seconds
Entry point: 150
Max level: 9


In [75]:
# Cell 52 - Benchmark 1K after fix
results_1000_fixed = []

for ef in [10, 20, 50, 100]:
    metrics = benchmark_hnsw(
        hnsw_1000_fixed,
        embeddings[:1000],
        ids[:1000],
        embeddings[:100],
        ef_search=ef,
        k=10
    )

    results_1000_fixed.append({
        "dataset_size": 1000,
        "ef_search": ef,
        "recall_at_10": metrics["recall"],
        "avg_latency_ms": metrics["avg_latency_ms"],
        "median_latency_ms": metrics["median_latency_ms"],
        "build_time_sec": build_time_fixed_1k
    })

    print(
        f"ef_search={ef} | "
        f"Recall@10={metrics['recall']:.2%} | "
        f"Avg latency={metrics['avg_latency_ms']:.3f} ms | "
        f"Median latency={metrics['median_latency_ms']:.3f} ms"
    )

pd.DataFrame(results_1000_fixed)

ef_search=10 | Recall@10=67.50% | Avg latency=0.297 ms | Median latency=0.273 ms
ef_search=20 | Recall@10=72.50% | Avg latency=0.369 ms | Median latency=0.343 ms
ef_search=50 | Recall@10=78.50% | Avg latency=0.565 ms | Median latency=0.534 ms
ef_search=100 | Recall@10=80.60% | Avg latency=0.773 ms | Median latency=0.775 ms


,dataset_size,ef_search,recall_at_10,avg_latency_ms,median_latency_ms,build_time_sec
0,1000,10,0.675,0.297114,0.27325,0.83918
1,1000,20,0.725,0.368758,0.34290,0.83918
2,1000,50,0.785,0.564885,0.53400,0.83918
3,1000,100,0.806,0.773439,0.77535,0.83918


In [76]:
# Cell 53 - Rebuild a fresh 10K HNSW index with the corrected implementation

hnsw_10k_fixed, build_time_fixed_10k = build_hnsw(
    embeddings[:10000],
    ids[:10000],
    M=8,
    ef_construction=50
)

print(f"10K build time: {build_time_fixed_10k:.3f} seconds")
print(f"Entry point: {hnsw_10k_fixed.entry_point}")
print(f"Max level: {hnsw_10k_fixed.max_level}")

10K build time: 9.071 seconds
Entry point: 9327
Max level: 16


In [77]:
# Cell 54 - Benchmark 10K after fix
results_10k_fixed = []

for ef in [10, 20, 50, 100]:
    metrics = benchmark_hnsw(
        hnsw_10k_fixed,
        embeddings[:10000],
        ids[:10000],
        embeddings[:100],
        ef_search=ef,
        k=10
    )

    results_10k_fixed.append({
        "dataset_size": 10000,
        "ef_search": ef,
        "recall_at_10": metrics["recall"],
        "avg_latency_ms": metrics["avg_latency_ms"],
        "median_latency_ms": metrics["median_latency_ms"],
        "build_time_sec": build_time_fixed_10k
    })

    print(
        f"ef_search={ef} | "
        f"Recall@10={metrics['recall']:.2%} | "
        f"Avg latency={metrics['avg_latency_ms']:.3f} ms | "
        f"Median latency={metrics['median_latency_ms']:.3f} ms"
    )

pd.DataFrame(results_10k_fixed)

ef_search=10 | Recall@10=31.80% | Avg latency=0.369 ms | Median latency=0.324 ms
ef_search=20 | Recall@10=37.70% | Avg latency=0.380 ms | Median latency=0.359 ms
ef_search=50 | Recall@10=43.20% | Avg latency=0.688 ms | Median latency=0.595 ms
ef_search=100 | Recall@10=45.00% | Avg latency=1.063 ms | Median latency=1.057 ms


,dataset_size,ef_search,recall_at_10,avg_latency_ms,median_latency_ms,build_time_sec
0,10000,10,0.318,0.368543,0.3240,9.070713
1,10000,20,0.377,0.380119,0.3586,9.070713
2,10000,50,0.432,0.688442,0.5951,9.070713
3,10000,100,0.450,1.063344,1.0567,9.070713


In [78]:
# Cell 55 - Final summary table after the HNSW fix

summary_df = pd.concat([
    pd.DataFrame(results_1000_fixed),
    pd.DataFrame(results_10k_fixed)
], ignore_index=True)

summary_df[["dataset_size", "ef_search", "recall_at_10", "avg_latency_ms", "median_latency_ms"]]

,dataset_size,ef_search,recall_at_10,avg_latency_ms,median_latency_ms
0,1000,10,0.675,0.297114,0.27325
1,1000,20,0.725,0.368758,0.34290
2,1000,50,0.785,0.564885,0.53400
3,1000,100,0.806,0.773439,0.77535
4,10000,10,0.318,0.368543,0.32400
5,10000,20,0.377,0.380119,0.35860
6,10000,50,0.432,0.688442,0.59510
7,10000,100,0.450,1.063344,1.05670
